# Data Manipulation — Intermediate
## Reshaping & Combining Data (R)

**Philosophy:** Assumes clean data and familiarity with the tidyverse API. Each
section covers a multi-step pattern — not individual function syntax, but how
to chain tools together to solve a real reshaping problem.

---

## Decision Table

| If your problem is... | Go to |
| :--- | :--- |
| Data is wide and you need one row per observation | §1 — Wide vs Long |
| Need top-N per group, conditional aggregation, or filter-then-agg | §2 — Advanced GroupBy |
| Joining 3+ tables, row count explodes after merge, or need anti-join | §3 — Multi-table Pipelines |
| Need to change time frequency, align two time series, or forward-fill after resample | §4 — Time-Series Reshaping |
| Column contains JSON, lists, or list-columns | §5 — Nested & List-like Data |
| Reading and combining many CSV or Excel files | §6 — Combining Multiple Files |

## Series Map

| Notebook | Use it for |
| :--- | :--- |
| `DM_Basic` | Cleaning & QC — audit, types, nulls, dupes, strings, outliers, column names |
| `DM_Intermediate` **(this note)** | Reshaping & combining — wide/long, group-by, multi-table joins, time-series, nested, many files |
| `DM_Advanced` | Feature engineering — encoding, transforms, binning, date, lag/window, interactions, text, leakage |
| `Reference_Tidyverse / Reference_BaseR` | Syntax lookups for individual functions |

> **Environment:** R 4.x / tidyverse 2.x. A small sample frame is defined below
> for runnable examples.

In [ ]:
library(dplyr)
library(tidyr)
library(lubridate)

df <- tibble::tibble(
  user_id    = c(1, 1, 2, 2, 3),
  region     = c('US','US','EU','EU','US'),
  platform   = c('iOS','Android','iOS','iOS','Web'),
  revenue    = c(100, 150, 200, 120, 90),
  event_date = ymd(c('2024-01-01','2024-02-01','2024-01-15','2024-03-01','2024-02-20'))
)
print(df)

---
## §1 — Wide vs Long Format

**Wide:** one row per entity, one column per time period or category. Easy to
read, hard to aggregate.
**Long:** one row per observation, categories stored as values in a column.
Required for most group-by/plotting/modeling operations.

In [ ]:
library(tidyr)
library(stringr)

# Wide format (one row per user, one column per quarter)
wide <- tibble::tibble(
  user_id    = c(1, 2, 3),
  Q1_revenue = c(100, 200, 150),
  Q2_revenue = c(120, 210, 160),
  Q3_revenue = c(130, 190, 170)
)
cat("Wide format:\n"); print(wide)

# Wide -> Long: pivot_longer
long <- wide |> pivot_longer(
  cols      = c(Q1_revenue, Q2_revenue, Q3_revenue),
  names_to  = "quarter",
  values_to = "revenue"
) |> mutate(quarter = str_remove(quarter, "_revenue"))
cat("\nLong format:\n"); print(long)

# Long -> Wide: pivot_wider
wide_again <- long |>
  pivot_wider(id_cols = user_id, names_from = quarter, values_from = revenue)
cat("\nBack to wide:\n"); print(wide_again)

# Multiple value columns: .value sentinel
wide2 <- tibble::tibble(
  user_id = c(1, 2),
  rev_Q1 = c(100, 200), cost_Q1 = c(60, 110),
  rev_Q2 = c(120, 210), cost_Q2 = c(70, 120)
)
df_long <- wide2 |> pivot_longer(
  cols = -user_id,
  names_to  = c(".value", "quarter"),
  names_sep = "_"
)
cat("\nMulti-value pivot_longer (.value sentinel):\n"); print(df_long)


**Common mistakes:**
- Forgetting `id_cols=` in `pivot_wider()` when extra grouping columns are still present — without it, dplyr may try to use every remaining column as part of the row key, producing more rows than expected
- `pivot_wider()` raises (or produces list-columns) with duplicate index/column pairs unless `values_fn=` is supplied — pass `values_fn = sum` (or another aggregator) explicitly when duplicates are possible, the direct analog to pandas needing `pivot_table` instead of `pivot`
- Forgetting that `pivot_longer`'s `.value` sentinel in `names_to` is what makes the stub-splitting pattern work — omitting it just produces one combined label column, not separate `rev`/`cost` columns

**Verified — the `.value` / `names_sep` pattern, the R analog to `pd.wide_to_long`:**

In [ ]:
wide2 <- tibble::tibble(user_id = c(1, 2),
                        rev_Q1 = c(100, 200), cost_Q1 = c(60, 110),
                        rev_Q2 = c(120, 210), cost_Q2 = c(70, 120))
out <- wide2 |> pivot_longer(cols = -user_id, names_to = c(".value", "quarter"), names_sep = "_")
print(out)

---
## §2 — Advanced GroupBy Patterns

Beyond basic `group_by() |> summarise()` — patterns for filtering groups,
selecting top-N within a group, and conditional aggregation. Syntax reference
is in the Tidyverse reference (§6); this section focuses on the multi-step
patterns.

In [ ]:
library(dplyr)
library(tidyr)

# Richer sample for group-by patterns (5 rows is too small to show most patterns)
set.seed(42)
df_grp <- tibble::tibble(
  user_id  = rep(1:20, each = 3),
  region   = sample(c("US","EU","APAC"), 60, replace = TRUE),
  platform = sample(c("iOS","Android","Web"), 60, replace = TRUE),
  revenue  = round(runif(60, 10, 500), 2),
  month    = rep(c("2024-01","2024-02","2024-03"), 20)
)

# Pattern 1: Top-N per group (top 3 users by revenue in each region)
top3 <- df_grp |>
  group_by(region) |>
  mutate(rev_rank = dense_rank(desc(revenue))) |>
  filter(rev_rank <= 3) |>
  select(region, user_id, revenue, rev_rank) |>
  ungroup()
cat("Top 3 by revenue per region:\n"); print(top3 |> arrange(region, rev_rank))

# Pattern 2: Filter entire groups by aggregate condition
active <- df_grp |>
  group_by(region) |>
  filter(sum(revenue) > 1000) |>
  ungroup()
cat(sprintf("\nRegions with total revenue > 1000: %s\n",
    paste(unique(active$region), collapse = ", ")))

# Pattern 3: Conditional aggregation
result <- df_grp |>
  group_by(region) |>
  summarise(
    total_rev   = sum(revenue, na.rm = TRUE),
    ios_rev     = sum(if_else(platform == "iOS", revenue, 0), na.rm = TRUE),
    unique_users = n_distinct(user_id),
    .groups = "drop"
  ) |>
  mutate(ios_share = round(ios_rev / total_rev, 3))
cat("\nConditional aggregation (iOS revenue share):\n"); print(result)

# Pattern 4: Group-level features back on each row
df_grp <- df_grp |>
  group_by(region) |>
  mutate(
    region_avg    = mean(revenue, na.rm = TRUE),
    pct_of_region = revenue / sum(revenue, na.rm = TRUE)
  ) |>
  ungroup()
cat("\nGroup features on original rows (first 6):\n")
print(df_grp |> select(user_id, region, revenue, region_avg, pct_of_region) |> head(6))

# Pattern 5: Custom aggregation (90th percentile)
result_p90 <- df_grp |>
  group_by(region) |>
  summarise(
    p90_rev = quantile(revenue, 0.9, na.rm = TRUE),
    cv_rev  = round(sd(revenue) / mean(revenue), 3),
    .groups = "drop"
  )
cat("\n90th percentile revenue per region:\n"); print(result_p90)


**Pattern selection:**

| Goal | Tool |
| :--- | :--- |
| Top-N per group | `group_by() |> mutate(dense_rank())` + filter, or `arrange + group_by() |> slice_head(n)` |
| Remove entire groups that fail a condition | `group_by() |> filter(aggregate_condition)` |
| Conditional sum/count within a group | `if_else()` to zero-out + `summarise()` |
| Group stats back on each row | `group_by() |> mutate()` |
| Custom aggregation (percentile, CV) | `summarise(stat = quantile(x, ...))` |

**Common mistakes:**
- Using `summarise()` when `mutate()` is needed — `summarise()` collapses rows and loses the join-back to original data, the same trap as pandas' `agg` vs `transform`
- Using `group_by() |> filter()` when you only want to flag — filter removes rows permanently; use `mutate()` to add a logical flag column instead
- Forgetting `ungroup()` after a grouped `mutate()` or `filter()` — the grouping silently persists into later pipe steps and corrupts subsequent aggregate computations

---
## §3 — Multi-table Pipelines

Single-join operations are in the Tidyverse reference (§5). This section covers
sequences of joins, diagnosing unexpected row count changes, and anti-join patterns.

In [ ]:
library(dplyr)

# Sample tables for all multi-table patterns in this section
set.seed(42)
users <- tibble::tibble(
  user_id = 1:5,
  name    = c("Alice","Bob","Carol","Dan","Eve"),
  region  = c("US","EU","US","APAC","EU")
)
orders <- tibble::tibble(
  order_id   = 1:8,
  user_id    = c(1,1,2,3,3,3,4,5),
  amount     = round(runif(8, 20, 200), 2),
  order_date = as.Date("2024-01-01") + sample(0:90, 8)
)
order_items <- tibble::tibble(
  order_id = rep(1:8, each = 2),
  item_id  = 1:16,
  price    = round(runif(16, 5, 80), 2)
)
tags <- tibble::tibble(
  user_id    = c(1,1,2,3,4),
  tag        = c("vip","loyal","new","loyal","vip"),
  created_at = as.Date("2024-01-01") + c(10,20,5,30,15)
)
churned_users <- tibble::tibble(user_id = c(2, 4))
user_roles <- tibble::tibble(user_id = c(1,2,3), role_id = c(1,2,1))
roles      <- tibble::tibble(role_id = c(1,2), role_name = c("admin","viewer"))

cat("Sample tables created: users, orders, order_items, tags, churned_users\n\n")

# ── Pattern 1: Sequential 3-table join ───────────────────────────────────────
result <- users |>
  left_join(orders,      by = "user_id") |>
  left_join(order_items, by = "order_id")
cat(sprintf("Sequential join: %d users -> %d order rows -> %d item rows\n",
    nrow(users), nrow(left_join(users, orders, by="user_id")), nrow(result)))

# ── Diagnosing row count explosion ───────────────────────────────────────────
cat("\norders per user_id:\n"); print(orders |> count(user_id))
cat("Max orders per user:", max(table(orders$user_id)), "(>1 means fan-out risk)\n")

# ── Pattern 2: Aggregate before joining to avoid fan-out ─────────────────────
orders_agg <- orders |>
  group_by(user_id) |>
  summarise(order_count = n(), total_spend = sum(amount), last_order = max(order_date), .groups = "drop")
result_1to1 <- users |> left_join(orders_agg, by = "user_id")
cat("\n1-to-1 join after aggregation:\n"); print(result_1to1)

# ── Pattern 3: Anti-join ─────────────────────────────────────────────────────
active_users <- users |> anti_join(churned_users, by = "user_id")
cat("\nActive users (anti-join removes churned):\n"); print(active_users)

# ── Pattern 4: Many-to-many -- aggregate tags first ──────────────────────────
tags_agg <- tags |>
  group_by(user_id) |>
  summarise(tag_list = paste(tag, collapse = ","), .groups = "drop")
result_tags <- users |> left_join(tags_agg, by = "user_id")
cat("\nUsers with tag list:\n"); print(result_tags)

# ── Pattern 5: Bridge table join ──────────────────────────────────────────────
result_roles <- users |>
  inner_join(user_roles, by = "user_id") |>
  left_join(roles, by = "role_id")
cat("\nUsers with roles:\n"); print(result_roles)


**Row count change after join — diagnostic guide:**

| Row count change | Likely cause | Fix |
| :--- | :--- | :--- |
| Count stays same | 1-to-1 match ✅ | — |
| Count increases | Right table has multiple rows per key | Aggregate right table first |
| Count decreases | Inner join dropped non-matching rows | Switch to `left_join` if needed |
| Count drops to near zero | Join key type mismatch (numeric vs character) | Cast keys to same type |

**Common mistakes:**
- Joining without checking key uniqueness — row explosion is silent and hard to detect later
- Joining on columns with different types — `*_join()` raises a type-mismatch **error** in R (unlike pandas, which silently produces zero matches), so this failure mode is actually caught earlier, but the fix is the same: cast both keys to the same type first
- Using `inner_join()` when you need all rows from the left table — rows drop silently

**Two §3 additions: join cardinality checks and `coalesce()` across frames.**

The `join_check` helper above prints row counts to *detect* fan-out after the
fact; dplyr's `*_join()` functions accept a `relationship =` argument (dplyr
1.1+) that **guards** against it directly, raising an error the moment a join
would violate the cardinality you expect — the direct equivalent of pandas'
`merge(..., validate=...)`. `coalesce()` fills gaps in a primary vector from a
backup, position-aligned (R has no separate `combine_first()` — `coalesce()`
covers both the single-vector and whole-frame cases).

In [ ]:
library(dplyr)
users  <- tibble::tibble(user_id = c(1, 2, 3), name = c('a','b','c'))
orders <- tibble::tibble(user_id = c(1, 1, 2), amt = c(10, 20, 30))   # user 1 appears twice

result <- tryCatch(
  left_join(users, orders, by = "user_id", relationship = "one-to-one"),
  error = function(e) {
    cat('relationship="one-to-one" ->', conditionMessage(e), '\n')
    NULL
  }
)
cat('relationship="one-to-many" -> rows:',
    nrow(left_join(users, orders, by = "user_id", relationship = "one-to-many")), '\n')

# coalesce -- fill NAs in primary from backup, position-aligned (R's combine_first equivalent)
primary <- c(1, NA, 3)
backup  <- c(9, 2,  9)
cat('coalesce(primary, backup):', coalesce(primary, backup), '\n')

---
## §4 — Time-Series Reshaping

Covers changing temporal granularity, filling gaps in time series, and aligning
two series that run on different frequencies. R has no single `.resample()`
method tied to a `DatetimeIndex` the way pandas does — instead, you derive a
period column with `lubridate::floor_date()` and `group_by()` on it, which is
slightly more verbose but makes the aggregation step fully explicit.

In [ ]:
library(dplyr)
library(lubridate)
library(tidyr)

# Add a parsed date column (event_date is already Date from SS0)
df_ts <- df |> mutate(date = event_date) |> arrange(date)

# Pattern 1: Downsample daily -> monthly
monthly <- df_ts |>
  mutate(period = floor_date(date, "month")) |>
  group_by(period) |>
  summarise(revenue = sum(revenue, na.rm = TRUE), users = n_distinct(user_id), .groups = "drop")
cat("Monthly summary:\n"); print(monthly)

# Pattern 2: Resample to weekly
weekly <- df_ts |>
  mutate(period = floor_date(date, "week")) |>
  group_by(period) |>
  summarise(revenue = sum(revenue, na.rm = TRUE), .groups = "drop")
cat("\nWeekly summary:\n"); print(weekly)

# Pattern 3: Create a complete date spine (fill gaps)
date_spine <- tibble::tibble(
  date = seq(min(df_ts$date), max(df_ts$date), by = "month")
)
complete_ts <- date_spine |>
  left_join(monthly |> rename(date = period), by = "date") |>
  mutate(revenue = replace_na(revenue, 0))
cat("\nComplete monthly spine (gaps filled with 0):\n"); print(complete_ts)


**Common mistakes:**
- Forward-filling before checking sort order — `fill()` fills in current row order; always `arrange()` by date first
- Reaching for a single `.resample()`-style call out of pandas habit — R's `floor_date() |> group_by() |> summarise()` pattern is the idiomatic replacement; there is no equivalent single function, but the explicit version makes the aggregation choice (`sum` vs `mean`) unmissable
- Downsampling with `mean()` when `sum()` is correct — e.g. daily revenue should be summed, not averaged, when going monthly, identical trap to pandas
- Building a date spine with `seq(..., by = "day")` but the original dates have time-of-day components — strip to date-only with `as_date()` first, the R equivalent of pandas' `.dt.normalize()`

---
## §5 — Nested & List-like Data

Common when working with API responses, JSON exports, or data where one cell
contains a list. R's native list-column support (any column can hold arbitrary
R objects, including other vectors) makes this somewhat more natural than
pandas' object-dtype columns, but flattening/exploding still needs the same
deliberate handling.

In [ ]:
library(dplyr)
library(tidyr)
library(jsonlite)

# Pattern 1: Flatten a list-column
df_list <- tibble::tibble(
  user_id  = c(1, 2, 3),
  metadata = list(
    list(source = "organic",  campaign = "summer"),
    list(source = "paid",     campaign = "winter"),
    list(source = "referral", campaign = "spring")
  )
)
meta_expanded <- df_list$metadata |> purrr::map_dfr(as_tibble)
df_flat <- df_list |> select(-metadata) |> bind_cols(meta_expanded)
cat("Flattened list-column:\n"); print(df_flat)

# Pattern 2: Unnest (explode) a list-column -- one row per element
df_tags <- tibble::tibble(
  user_id = c(1, 2),
  tags    = list(c("python", "sql"), c("r", "tidyverse", "ggplot2"))
)
cat("\nBefore unnest:\n"); print(df_tags)
df_exploded <- df_tags |> unnest(tags)
cat("After unnest (one row per tag):\n"); print(df_exploded)

# Pattern 3: Parse stringified JSON
df_json <- tibble::tibble(
  id   = c(1, 2),
  data = c('{"score": 85, "grade": "B"}', '{"score": 92, "grade": "A"}')
)
df_parsed <- df_json |>
  mutate(parsed = purrr::map(data, jsonlite::fromJSON)) |>
  tidyr::unnest_wider(parsed)
cat("\nParsed JSON strings:\n"); print(df_parsed)


**Common mistakes:**
- Reaching for `eval(parse(text = ...))` to "parse" a stringified structure — this is R's `eval()` equivalent and carries the same security risk as Python's `eval()`; if the string is JSON-like, parse it with `jsonlite::fromJSON()` instead, never by evaluating it as code
- Calling `jsonlite::fromJSON()` on a column with `NA` values without checking first — raises an error; filter or replace `NA` with an empty JSON object string (`"{}"`) first
- Forgetting that `unnest()` requires the column to actually be a **list-column** (`is.list(df$col)` is `TRUE`) — a plain character column holding text that merely *looks* like a list needs Pattern 3's parsing step first

---
## §6 — Combining Multiple Files

Common in real-world data pipelines: monthly data exports, partitioned data
dumps, or multi-sheet Excel files. The challenge is schema consistency across
files.

In [ ]:
library(dplyr)
library(purrr)
library(readr)

# Create sample CSV files in a temp directory so this cell is self-contained
tmp_dir <- tempdir()
months  <- c("2024-01", "2024-02", "2024-03")

# Write three monthly CSV files
for (m in months) {
  monthly_df <- tibble::tibble(
    user_id = sample(1:100, 20),
    date    = m,
    revenue = round(runif(20, 10, 500), 2)
  )
  write_csv(monthly_df, file.path(tmp_dir, paste0(m, ".csv")))
}

# ── Pattern 1: Read and combine many CSVs ────────────────────────────────────
files <- list.files(tmp_dir, pattern = "\\.csv$", full.names = TRUE)
cat("CSV files found:", basename(files), "\n\n")

df_combined <- files |> map_dfr(read_csv, show_col_types = FALSE)
cat(sprintf("Combined: %d rows from %d files\n", nrow(df_combined), length(files)))
cat("Columns:", names(df_combined), "\n\n")

# ── Pattern 2: Add source file column ────────────────────────────────────────
df_sourced <- files |> map_dfr(function(f) {
  chunk <- read_csv(f, show_col_types = FALSE)
  chunk$source_file <- tools::file_path_sans_ext(basename(f))
  chunk
})
cat("With source column (first 4 rows):\n")
print(head(df_sourced, 4))

# ── Pattern 3: Validate schema before combining ───────────────────────────────
dfs <- files |> map(~ read_csv(.x, show_col_types = FALSE))
expected_cols <- names(dfs[[1]])
all_match <- all(map_lgl(dfs[-1], ~ setequal(names(.x), expected_cols)))
cat(sprintf("\nSchema consistent across all files: %s\n", all_match))

# ── Pattern 4: bind_rows() with mismatched schemas ───────────────────────────
df_extra <- tibble::tibble(user_id = 999, date = "2024-04", revenue = 100, extra_col = "new")
df_combined_outer <- bind_rows(df_combined, df_extra)
cat(sprintf("\nbind_rows() with extra column: %d rows x %d cols (extra_col filled with NA)\n",
    nrow(df_combined_outer), ncol(df_combined_outer)))
print(tail(df_combined_outer, 2))


**Common mistakes:**
- Assuming all files have the same schema — always validate before combining, especially with data from different time periods
- Loading all columns from all files when only a few are needed — use `col_select` to limit memory usage
- Using `bind_rows()` with mismatched types across files (e.g. `integer` in one file, `double` in another) — readr/dplyr will coerce, but check `glimpse()` after combining to verify the result is what you expect, same discipline as checking `df.dtypes` after `pd.concat`
- Forgetting that `bind_rows()` has no row-index concept to manage — unlike pandas' `pd.concat` (where `ignore_index=True` is an easy-to-forget but essential argument), tibbles simply don't carry duplicate row indices in the first place, so there is nothing to remember here

---
## Decision Guide

```
Need to change data shape?
+-- Wide -> Long (many columns -> one value column)  -> pivot_longer(cols, names_to, values_to)   (§1)
+-- Long -> Wide (categories -> separate columns)    -> pivot_wider(id_cols, names_from, values_from) (§1)
\-- Multi-value wide -> long                         -> pivot_longer(names_to = c(".value", ...))  (§1)

Need advanced group-by?
+-- Top-N per group                                  -> group_by() |> mutate(dense_rank()) + filter (§2)
+-- Remove groups failing a condition                -> group_by() |> filter(aggregate_condition)   (§2)
+-- Conditional sum within group (SUM CASE WHEN)     -> if_else() + group_by() |> summarise()        (§2)
+-- Group stats back on each row                     -> group_by() |> mutate()                       (§2)
\-- Custom aggregation (percentile, ratio)           -> summarise(stat = quantile(x, ...))           (§2)

Combining multiple tables?
+-- Row count grows after join                       -> aggregate right table first                  (§3)
+-- Rows in A with no match in B                      -> anti_join()                                  (§3)
+-- Many-to-many relationship                         -> aggregate one side or use a bridge table      (§3)
\-- Key type mismatch (rows drop to zero / error)    -> cast both keys to same type                  (§3)

Working with time series?
+-- Change frequency (daily -> monthly)               -> floor_date() |> group_by() |> summarise()    (§4)
+-- Fill gaps in time series                          -> build date spine + left_join                 (§4)
+-- Carry last known value forward                    -> fill(.direction = "down")                    (§4)
\-- Align different frequencies                      -> left_join + fill                              (§4)

Column contains lists / JSON?
\-- Flatten nested JSON                              -> jsonlite::fromJSON(flatten = TRUE)            (§5)

Column contains list-columns?
+-- One row per list element                          -> unnest(col)                                  (§5)
+-- Fixed-length list -> separate columns             -> purrr::map_dfr() + bind_cols()                (§5)
\-- Stringified JSON list                            -> jsonlite::fromJSON() (never eval/parse text)  (§5)

Reading many files?
+-- Same schema, just combine                         -> dir_ls() + map_dfr(read_csv)                 (§6)
+-- Track source file                                 -> add 'source_file' column per chunk            (§6)
+-- Schema may differ across files                    -> validate columns first, bind_rows() to align (§6)
\-- Multiple Excel sheets                            -> map over excel_sheets() + bind_rows(.id=)     (§6)
```